In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm, trange
from collections import defaultdict

from libs.DataLoader import Loader
from libs.constants import *
from libs.utils import NDCG

In [2]:
def get_equal_weights(indices):
    n = len(indices)
    return np.ones(n) / n

def get_linear_weights(indices):
    n = len(indices)
    return (np.argsort(np.argsort(indices)) + 1) / (n * (n + 1) / 2)

def get_exponential_weights(indices):
    ranks = np.argsort(np.argsort(indices)) + 1
    return np.exp(ranks) / np.sum(np.exp(ranks))

def add_to_top_lists(val_item, agg_user, score, top_per_item, max_per_user):
    if val_item not in item_top_lists:
        item_top_lists[val_item] = []

    heap = item_top_lists[val_item]

    if user_selection_counts[agg_user] >= max_per_user:
        return

    if len(heap) < top_per_item:
        heapq.heappush(heap, (score, agg_user))
        user_selection_counts[agg_user] += 1
    else:
        if score > heap[0][0]:
            old_score, old_user = heapq.heappop(heap)
            user_selection_counts[old_user] -= 1

            heapq.heappush(heap, (score, agg_user))
            user_selection_counts[agg_user] += 1

In [3]:
loader = Loader('ur0.01', content_embedding_size=8, batch_size=500_000)
(train_df, val_df), users_df, items_df = loader.load_data(convert_to_pandas=True)
items_df.set_index(ITEM, inplace=True)
users_df.set_index(USER, inplace=True)

Load metadata
Create lazy interaction datasets
Get unique users/items


  0%|          | 0/753 [00:00<?, ?it/s]

  0%|          | 0/34 [00:00<?, ?it/s]

Filter embeddings
Process metadata
Compute aggregates


  0%|          | 0/753 [00:00<?, ?it/s]

Filter interactions
Finalize interactions
Count
Train: 13_927_019 Val: 2_797_438 Users: 22_469 Items: 12_012_895
Convert to pandas


In [4]:
train_df: pd.DataFrame = train_df[train_df[TARGET] > 0]  # Interaction: positive
val_df: pd.DataFrame = val_df[val_df[TARGET] > 0]  # Interaction: positive

print(f"Train: {len(train_df):_} Val: {len(val_df):_}")

Train: 13_900_707 Val: 473_197


In [5]:
num_recent_videos = 50

agg = train_df \
      .sort_values([USER, TIME_INDEX], ascending=[True, False]) \
      .groupby(USER) \
      .agg({ITEM: lambda x: list(x.head(num_recent_videos)),
            TIME_INDEX: lambda x: list(x.head(num_recent_videos))})

val = pd.DataFrame({ITEM: val_df[ITEM].unique()})
cold_val_items = set(val[ITEM].unique()) - set(train_df[ITEM].unique())

In [22]:
import heapq

agg_batch_size = 500
val_batch_size = 5000
top_per_item = 100
max_per_user = 101

n_agg = len(agg)
n_val = len(val)

item_top_lists = {}
user_selection_counts = defaultdict(int)

val_bar = trange(0, n_val, val_batch_size, position=0, desc="Val")
agg_bar = tqdm(total=(n_agg / agg_batch_size).__ceil__(), desc="Agg", position=1)
for val_batch_start in val_bar:
    val_batch_end = min(val_batch_start + val_batch_size, n_val)

    val_items = list(set().union(val.iloc[val_batch_start:val_batch_end][ITEM]))
    val_embeddings = np.vstack(items_df.loc[val_items, EMBEDDING])
    val_norms = np.linalg.norm(val_embeddings, axis=1, keepdims=True)

    agg_bar.n = 0
    agg_bar.last_print_n = 0
    for agg_batch_start in range(0, n_agg, agg_batch_size):
        agg_batch_end = min(agg_batch_start + agg_batch_size, n_agg)

        agg_items = list(set().union(*agg.iloc[agg_batch_start:agg_batch_end][ITEM]))
        agg_embeddings = np.vstack(items_df.loc[agg_items, EMBEDDING])
        agg_norms = np.linalg.norm(agg_embeddings, axis=1, keepdims=True)
        similarity = np.dot(val_embeddings, agg_embeddings.T) / (val_norms * agg_norms.T)

        agg_item_to_index = {item_id: idx for idx, item_id in enumerate(agg_items)}

        for i, agg_idx in enumerate(range(agg_batch_start, agg_batch_end)):
            row = agg.iloc[agg_idx]
            agg_user = row.name
            row_indices = [agg_item_to_index[item] for item in row[ITEM]]
            row_similarities = similarity[:, row_indices]

            weights = get_linear_weights(row[TIME_INDEX])
            user_scores = np.average(row_similarities, weights=weights, axis=1)

            for j, val_item in enumerate(val_items):
                score = user_scores[j]
                add_to_top_lists(val_item, agg_user, score, top_per_item, max_per_user)

        agg_bar.update(1)

results = []
for item, heap in item_top_lists.items():
    sorted_users = [user for score, user in sorted(heap, reverse=True)]
    results.append({
        ITEM: item,
        USER: sorted_users
    })

predict = pd.DataFrame(results)

Val:   0%|          | 0/40 [00:00<?, ?it/s]

Agg:   0%|          | 0/45 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [49]:
for pred, pred_name in [
    (predict, "constrained")
]:
    for true, true_name in [
        (val_df, "all"),
        (val_df[val_df[ITEM].isin(cold_val_items)], "cold")
    ]:
        print(f"Predict: {pred_name} Test: {true_name} NDCG: {NDCG(pred, true.groupby(ITEM).agg({USER: list})):.4f}")

Predict: constrained Test: all NDCG: 0.0507
Predict: constrained Test: cold NDCG: 0.0873
